# 00 · Exploración de Multimodal3DIdent

Objetivo: conocer la distribución de los factores generativos, el vocabulario de las descripciones de referencia y fijar los valores discretos que el extractor de atributos deberá reconocer.

Requiere haber corrido `python scripts/build_manifest.py`. Este notebook solo **lee** datos; ninguna figura del informe se genera aquí.

In [ ]:
%load_ext autoreload
%autoreload 2
from collections import Counter
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from vlmfid.data import data_root, read_manifest, disagreement, discrete_text_attributes

ROOT = data_root("base")

In [ ]:
pl_df = read_manifest("test")                     # Parquet tal cual (polars)
df = pd.DataFrame(pl_df.to_dicts())                # pandas para graficar, sin depender de pyarrow
ATTRS = discrete_text_attributes(pl_df)            # latentes de texto discretos detectados
print(len(df), "muestras de test | atributos discretos:", ATTRS)
df.head(3)

## Desacuerdo imagen vs texto
Techo alcanzable: si el caption dice un valor distinto del que tiene la imagen, un modelo fiel a la imagen no puede acertar contra ambas verdades a la vez.

In [ ]:
disagreement(pl_df)

## Factores discretos del texto
Base de la exactitud por atributo: cada valor debe poder recuperarse desde la descripción.

In [ ]:
fig, axes = plt.subplots(1, len(ATTRS), figsize=(4 * len(ATTRS), 3))
for ax, f in zip(axes, ATTRS):
    df[f"text_{f}"].value_counts().sort_index().plot.bar(ax=ax, title=f)
plt.tight_layout()

## Factores continuos de la imagen
Ángulos y tonos (hue): no se describen explícitamente en el texto, pero condicionan la dificultad visual.

In [ ]:
meta = {"id", "image_id", "image_path", "caption_ref", "split", "variant", "row_idx"}
cont = [c for c in df.columns if c not in meta and not c.startswith("text_") and df[c].nunique() > 10]
df[cont].hist(bins=40, figsize=(12, 6)); plt.tight_layout()

## Relación texto ↔ factores
¿Qué palabras aparecen para cada valor? Esto alimenta directamente el diccionario del extractor de atributos.

In [ ]:
def words_by_value(factor, top=8):
    out = {}
    for v, g in df.groupby(f"text_{factor}"):
        c = Counter(w.strip('.,').lower() for cap in g.caption_ref for w in cap.split())
        out[v] = [w for w, _ in c.most_common(40) if w not in {"a","the","is","of","in","image","at","an"}][:top]
    return pd.Series(out)

for f in ATTRS:
    print(f"\n== {f}"); print(words_by_value(f).to_string())

In [ ]:
df["n_words"] = df.caption_ref.str.split().str.len()
print(df.n_words.describe())
df.caption_ref.sample(10, random_state=0).tolist()

## Muestras visuales

In [ ]:
sample = df.sample(8, random_state=1)
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for ax, (_, r) in zip(axes.flat, sample.iterrows()):
    ax.imshow(Image.open(ROOT / r.image_path)); ax.axis("off")
    ax.set_title(r.caption_ref, fontsize=8, wrap=True)
plt.tight_layout()